# Joseph's addition.

### Disclaimer to the marker.

Note my group did not ask for any of my input on the project despite my repeated asking about how I could help and barred me from making any changes to their submission of the model, and report. Not only this but removed my name from the project.

Thus this serves as a correction to their modelling, which was not well thoughtout and AI laden (in the unsupervised way as please note I used ai for debugging).

**Central question of this addition.**

I will focus on asking, "What is the total number of flu cases 1 week ahead given all the past data?"



## Abstract

**Basic models**

| Methods | In Sample RMSE | Holdout RMSE |
| :-----| -----: | ----: |
Persistence RMSE | 4591.421 | 13862.3583 |
Average RMSE | 24425.026 | 70756.748 |
IID RMSE | 22315.5799 |  67731.8984 |
Holt-Winters (Additive) | 28896.9937 | 17400.1611 |


Of the simple models the persistence performed the best followed by Holt winters Additive and then Average.

**Complex models**:

| Methods | Holdout RMSE |
| :---- | ----:|
| Multi seasonal arima (lags at 1,19,52) | 8997.34 |
| Multi seasonal arima external regressors (lags at 1,19,52) | Holdout: 10295.25 |
| Multi seasonal arima external regressors extended (lags at 1,19,52) | Holdout: 9823.528 |
| STL | Holdout:12592.93 |
| STL + MS ARIMA $(1,0,3)(0,1,0)_{19}(1,0,2)_{52}$ | Holdout: 9685.038 | 
|ARIMA $(2,0,3)(3,1,1)_{52}$ | 8726.74 |

The ARIMA model performed the best forecast on the holdout set. It explained the data to a significant extent such that it the Ljung Box test fails to reject the residuals of the model as not white noise. Equally important, is that there is interaction in our model between the different age groups in our data. This interaction comes in the form of different age groups being drivers for ili numbers in other groups. More analysis is needed in this area. This effect however is so strong that it makes ignoring it worse than modelling simply the ili_total as whole. 



## Variables

The variables used were, ili_total, age_0_4, age_5_24, age_25_49, age_50_64, age_25_64, age_65, non_ili_total.
An ili case is defined by the CDC as a a fever above 100 fahrenheit and a cough or sore throat. Thus it captures many cases of flu, common colds and other things like SARS, Covid-19, Swine flu, and even pneumonia. The ili_totaal is then also subcatagorised by age group such that we can see the number of flu cases by age group.
The non_ili_total is the number of patients a provider sees that were not ili cases. 

### Theoretical analysis and data exploration.


**The data generation process**

Providers collect the number of flu cases and each week submit this data to the CDC. Due to this being optional the number of providers fluctuates each week. Note if at any time we have no providers we will by definition have no flu cases being recorded.
This can be seen at various times such as during 1999 week 21 to 39. In which too flu cases were recorded by any providers to neccessitate reporting on the number of flu cases to the CDC. This does not mean that no flu cases were observed just that it was not reported. In times of low flu cases for a particalar provider the provider will typically bother to submit the number of flu cases to the CDC.

**Structure of the data**

Another notable thing about our data is the different age groups. The total number of recorded flu cases each week is split into several variables. That then are subdivide again in 2009 week 40 to account for a richer interpretation of the data.
**What do I mean by this?**
In 2009 week 40 the CDC, and providers began providing data on the number of ili cases in the age catagory 25 to 49 and 50 to 64 whereas previously this was reported as one homogenous group, age 25 to 64.


### Transformations

So we note we have to transform the data. 

**Why**?

Well because trying to deal with percentages, and count data does not make any sense for forcasting models without transformations. This is because we will get possibly negative estimates for our forecast (with expoential smoothing and ARIMA), which is so obviously an egregious model error... 

So my model will work slightly differently but we must note these formulas.

percent_unweighted_ili = ili_total/total_patients * 100.

This is just a fact, in the way the data is produced.

Now we define:

non_ilitotal = total_patients - ili_total

The we will need to adjust the this percent_unweighted_ili to calculate the percent_weighted_ili from the unweighted percent by state and then adjusting for population size to get for the whole of the US. This formula and determininstic, but we do not have access to that data so trying to guess it is rather silly. Due to this we will not try to forecast it and instead only focus on forecasting the ili_total.

Lastly we note that we will not neccessarily know the number of provides in the future time period, (even though the CDC has markov models to determine this very accurately). Thus we will try to predict this too.

I wanted to try and best forecast the variable ili_total. As this has the most significance to an actual flu season. To do this I used several methods, Multi seasonal ARIMA, STL and STL + MS ARIMA.
Also note that we have several count data points, thus a natural idea is to try and model these as poisson random variables with a rate parameter that fluctuates, for each of these processes. The problem is that we are unable to actaully observe the rate parameter at each time. Thus we will work with the count data in log scale by taking log(data+1) for the count data.

Please note that in these predictions all the data pre 2002 week 40 was left out due to the necessity of reduction in model complexity involved in data censorsing.
MS ARIMA used due to the fact that after differencing there are significant lags at 19 weeks and 52 weeks, see Appendix. Notet the lag at 33 weeks is a reflection of the lag at 19 (52-19=33). This is indicative of the flu season lasting around 19 weeks but only happening once a year.

**Periods of interest**

There are some periods when the number of flu cases drops or changes significantly, and the relationship in the data changes. There are two main classes of periods of interst.

**Zero count periods:** 

This is a really significant problem. Our data sometimes shows zero flu counts. This is actually just a censoring. This happens when the flu count is to low so some providers stop reporting on the flu data. To combat this, noting that it does not happen at all anymore, we have started our models from the point at which there are the last sets of flu cases being zero, as to try and debias our models to the training set. Note that the low flu not being zero again after 2002 week 40 is also possibly due to the CDC having a bigger program to collect this data, having more established relationships with providers ect. In practice it may be different for each provider and the censoring rule for any particular provider may change over time and could also be slightly random. That being said a good model would try to approximate it at any given time. Also note that technically we have not removed the censoring problem fully as during any period there may be a number of providers censoring (since the provider count also fluctuates) however the the fix we have instituted does help our models. 

**Epidemic periods:**

This happens there is a mass spreading of a disease through a population. This changes the behaviour of individuals leading to different flu counts breaking seasonality in the data. This can be seen post the 2009 Swine flu Epidemic, the Covid 19 Epidemic, and not the 2003 SARS Epidemic as it never became wide spead in the US. These periods typically look very different than our regurlar seasonal structure. For instance the lockdowns during Covid lowered the number of ili cases post Covid.



Periods of interest in the data 
![Alt Text](./periods_of_interest.png "Periods of interest in the data")



### Plots

The ACF and PACF of ili_total are clearly non-stationary thus differencing will be required.

![Alt Text](./acf_pacf_log_ili_total_before_diff.png "ACF and PACF of log ili_total")

Note after differening that all variables have the same lag structure. See appendix. This suggest that the data is highly co-linear in logspace, and that it exhibits a a clear seasonal pattern, however after looking at the sliding acf (see appendix) this representation is less clear. A clearer picture of this is needed this could be in the form of a moving average of correlations to see if the effects are there at all times that is the driving nature is always present in the same degree or if (the more likely of the two) driving depends on spikes and that causes the data to move together ie an outbreak amoung children age_0_5 and age_5_24 will spread into adults age_25_64 and then seniors age_65 (Noting that it is about how which group gets sick first makes it a driver rather than the inherentness of the groups nature).

To further see the periodicity in the data a periodogram was used to see the frequency of oscillations resulting from the fourier transform. Note the big spike at 1.

Periodogram of the ili total.
![Alt Text](./Periodogram_ilitoal.png "Periodogram of the ili total")


### Outline of analysis

The goal was to predict the ili_total using three different methods. Note I did not explicitly use ARIMAX as my collegues were unsucessful in their endavour to do so.

Method one is running models on only the data from ili_total.

Method two was running models on the subgroups of ili_total (by age), and then summing the predictions to recover a prediction for ili_total, as its value is the sum of each of the cases by age group.

We will need a table here.

| Method | Model | Holdout RMSE |
| :-------- | -------- | ----: |
| ARIMA   | ARIMA $(2,0,3)(3,1,1)_{52}$   | 8726.74 |
| MS ARIMA   | MS ARIMA $(2,0,3)(2,0,2)_{19}(3,1,1)_{52}$  | 8997.34 | 
| STL  | periodic | 12592.93 | 
| STL + MS ARIMA   | MS ARIMA $(1,0,3)(0,1,0)_{19}(1,0,2)_{52}$   | 9685.038 | 


Method three was running models on the extended list of subgroups of ili_total (by age) noting the split that happened in the data of the subgroup age_25_64 into age_25_49 and age_49_64 at 2009 week 40.

We only did this with MS ARIMA as it performed the best.

| Group | Model | Holdout RMSE |
| :-------- | -------- | ----: |
| age_0_4   | MS ARIMA $(5,1,4)(0,0,0)_{19}(2,0,2)_{52}$   | 1890.60 |
| age_5_24   | MS ARIMA $(5,1,4)(0,0,0)_{19}(2,0,2)_{52}$  | 4391.80 | 
| age_25_49  | MS ARIMA $(1,1,2)(2,1,0)_{19}(2,0,1)_{52}$ | 2445.48 | 
| age_50_64  | MS ARIMA $(1,1,2)(2,1,0)_{19}(2,0,1)_{52}$ | 1153.19 | 
| age_65  | MS ARIMA $(1,0,0)(0,0,1)_{19}(1,1,1)_{52}$ | 1515.54 | 
| age_25_64   | MS ARIMA $(1,0,1)(1,1,0)_{19}(2,0,1)_{52}$   | 3966.20 | 

Resulting in

| Group | Model | Holdout RMSE |
| :-------- | -------- | ----: |
| Single | MS ARIMA $(2,0,3)(2,0,2)_{19}(3,1,1)_{52}$| 8997.34 |
| External | Joint | 10295.25 | 
| External Extended | Joint | 9823.52 |


![Alt Text](./stl_stlarima_forecast_plot.png "STL + ARIMA Models")
![Alt Text](./msarima_forecast_comparison.png "MS ARIMA Models")

Thus we deduce that interestingly the general ARIMA $(2,0,3)(3,1,1)_{52}$ performed the best followed by the single MS ARIMA.



![Alt Text](./stl_stlarima_forecast_plot.png "STL + ARIMA Models")
![Alt Text](./msarima_forecast_comparison.png "MS ARIMA Models")

Note that for the Box-Ljung test results for the residuals on ARIMA $(2,0,3)(3,1,1)_{52}$ were

|Test | Box-Ljung test| 
| :---- | :----: |
|data: | residuals |
| X-squared = 61.388, df = 52 | p-value = 0.1749 |

This suggests that the model has suffiecently explained the data.

![Alt Text](./Model_diagnostics_arima.png "Diagnostics of residuals ARIMA (2,0,3)(3,1,1)_{52}")

This is further agreed with by the following plots, which further indicate stationarity.

![Alt Text](./variogram_qqnorm_residuals_arima.png "Variogram and QQ Norm ARIMA Residuals")

The only problem is that we still have slightly leading lags from our residuals. This means that our residuals still have predictive power on our solution. This means we still have room for future improvement on our model. Note that this the ARIMA model still performed the best of all of the perturbations. Thus this is unlikely to be a simple fix. Despite this I have included this plot for the sake of academic honesty.

![Alt Text](./arima_residuals_leading.png "ARIMA residuals leading")

## Reference

I thank Harry Joe for the code used to produce some of the plots. Specifically the variogram and periodogram. The notes for the STAT 443 course weer also heavily used throughout this material.

I thank Harry Joe for the code used to produce some of the plots. Specifically the variogram and periodogram. The notes for the STAT 443 course weer also heavily used throughout this material.
Note, part of the data cleaning was done by one of my collegues, Michael Güntert.

All of the other work and work in this document as well as other files was done by me.


## Appendix

Here are the difference at lag 1 of the log of different age groups. Notice the spikes at lag 1,19,33,52 in both the ACF and the PACF, the sliding ACF at lag 19 and lag 52, and the MS ARIMA $(2,0,3)(2,0,2)_{19}(3,1,1)_{52}$ plots of the residuals.

![Alt Text](./Appendix_acf_pacf_diff.png "ACF and PACF of variables after diff log")

![Alt Text](./sliding_acf_lag19_lag52.png "Sliding ACF after diff log")
![Alt Text](./msarima_single_residuals.png "MS ARIMA (2,0,3)(2,0,2)_{19}(3,1,1)_{52}")
